# Notebook 19 — Temporal Spectral Dynamics of Residue Manifolds

**Repo:** `mod30-residue-lanes`

Notebook 19 studies rolling eigenspace stability and temporal phase structure across the eight admissible mod30 residue lanes.

Constraint view:

> Rolling prime-lane manifolds evolve through stable but non-identical spectral phases.


## Goals

1. Generate rolling prime-filtered residue manifolds.
2. Compute rolling covariance and eigenspace decompositions.
3. Track explained variance trajectories.
4. Measure eigenspace drift and mode rotation.
5. Compute spectral entropy.
6. Detect temporal phase structure.
7. Export figures, CSVs, JSON, and reports.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODULUS = 30
ADMISSIBLE_RESIDUES = [1,7,11,13,17,19,23,29]

print("REPO_ROOT:", REPO_ROOT)


In [ ]:
def prime_sieve(n_max):
    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    limit = int(np.sqrt(n_max)) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n_max+1:p] = False

    return sieve


## Rolling prime lane vectors

In [ ]:
N_MAX = 120000
WINDOW_SIZE = 3000
STEP_SIZE = 300

values = np.arange(1, N_MAX + 1)
prime_mask = prime_sieve(N_MAX)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_prime"] = prime_mask[values]
df["valid"] = df["is_admissible"] & df["is_prime"]

lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]

rows = []

for start in range(1, N_MAX - WINDOW_SIZE + 2, STEP_SIZE):
    stop = start + WINDOW_SIZE - 1

    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    prime_window = window[window["valid"]]

    counts = (
        prime_window
        .groupby("residue")
        .size()
        .reindex(ADMISSIBLE_RESIDUES, fill_value=0)
    )

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_center": (start + stop) / 2,
    }

    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)

    rows.append(row)

rolling_df = pd.DataFrame(rows)

rolling_csv = RESULTS_DIR / "notebook19_rolling_vectors.csv"
rolling_df.to_csv(rolling_csv, index=False)

rolling_df.head()


## Rolling spectral decomposition

In [ ]:
X = rolling_df[lane_cols].to_numpy(dtype=float)

ROLLING_BLOCK = 20

spectral_rows = []
eigenvector_store = []

for idx in range(ROLLING_BLOCK, len(X)):
    block = X[idx-ROLLING_BLOCK:idx]

    centered = block - block.mean(axis=0, keepdims=True)

    cov = np.cov(centered, rowvar=False)

    eigvals, eigvecs = np.linalg.eigh(cov)

    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    explained = eigvals / eigvals.sum()

    entropy = -np.sum(explained * np.log(explained + 1e-12))

    row = {
        "window_index": idx,
        "window_center": rolling_df.loc[idx, "window_center"],
        "spectral_entropy": float(entropy),
    }

    for k in range(4):
        row[f"mode_{k+1}_variance"] = float(explained[k])

    spectral_rows.append(row)

    eigenvector_store.append({
        "index": idx,
        "eigvecs": eigvecs,
        "explained": explained,
    })

spectral_df = pd.DataFrame(spectral_rows)

spectral_csv = RESULTS_DIR / "notebook19_rolling_spectral_metrics.csv"
spectral_df.to_csv(spectral_csv, index=False)

spectral_df.head()


## Eigenspace drift and rotation

In [ ]:
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

drift_rows = []

for i in range(1, len(eigenvector_store)):

    prev = eigenvector_store[i - 1]["eigvecs"]
    cur = eigenvector_store[i]["eigvecs"]

    mode1_prev = prev[:, 0]
    mode1_cur = cur[:, 0]

    mode2_prev = prev[:, 1]
    mode2_cur = cur[:, 1]

    sim1 = abs(cosine_similarity(mode1_prev, mode1_cur))
    sim2 = abs(cosine_similarity(mode2_prev, mode2_cur))

    theta1 = np.degrees(np.arccos(np.clip(sim1, -1, 1)))
    theta2 = np.degrees(np.arccos(np.clip(sim2, -1, 1)))

    drift_rows.append({
        "window_index": eigenvector_store[i]["index"],
        "mode1_similarity": sim1,
        "mode2_similarity": sim2,
        "mode1_rotation_deg": theta1,
        "mode2_rotation_deg": theta2,
    })

drift_df = pd.DataFrame(drift_rows)

drift_csv = RESULTS_DIR / "notebook19_eigenspace_drift.csv"
drift_df.to_csv(drift_csv, index=False)

drift_df.head()


## Temporal similarity heatmap

In [ ]:
transition_matrix = np.zeros((len(X), len(X)))

for i in range(len(X)):
    for j in range(len(X)):
        transition_matrix[i, j] = cosine_similarity(X[i], X[j])

transition_csv = RESULTS_DIR / "notebook19_transition_similarity_matrix.csv"
pd.DataFrame(transition_matrix).to_csv(transition_csv, index=False)


## Low-dimensional phase embedding

In [ ]:
X_centered = X - X.mean(axis=0, keepdims=True)

cov = np.cov(X_centered, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)

order = np.argsort(eigvals)[::-1]
eigvecs = eigvecs[:, order]

embedding = X_centered @ eigvecs[:, :2]

embedding_df = pd.DataFrame({
    "x": embedding[:,0],
    "y": embedding[:,1],
    "window_center": rolling_df["window_center"],
})

embedding_csv = RESULTS_DIR / "notebook19_phase_embedding.csv"
embedding_df.to_csv(embedding_csv, index=False)

embedding_df.head()


## Summary exports

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "19_lane_residue_19",
    "title": "Temporal Spectral Dynamics of Residue Manifolds",
    "modulus": MODULUS,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "rolling_windows": int(len(rolling_df)),
    "rolling_blocks": int(len(spectral_df)),
    "mean_spectral_entropy": float(spectral_df["spectral_entropy"].mean()),
    "mode1_mean_variance": float(spectral_df["mode_1_variance"].mean()),
    "mode2_mean_variance": float(spectral_df["mode_2_variance"].mean()),
    "mean_mode1_rotation_deg": float(drift_df["mode1_rotation_deg"].mean()),
    "mean_mode2_rotation_deg": float(drift_df["mode2_rotation_deg"].mean()),
}

summary_path = RESULTS_DIR / "notebook19_temporal_spectral_summary.json"

summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)

# Rolling explained variance
plt.figure(figsize=(12,5))
plt.plot(spectral_df["window_center"], spectral_df["mode_1_variance"], label="mode1")
plt.plot(spectral_df["window_center"], spectral_df["mode_2_variance"], label="mode2")
plt.plot(spectral_df["window_center"], spectral_df["mode_3_variance"], label="mode3")
plt.xlabel("Window center")
plt.ylabel("Explained variance")
plt.title("Notebook 19 — Rolling Explained Variance")
plt.legend()
savefig("notebook19_rolling_explained_variance.png")

# Spectral entropy
plt.figure(figsize=(12,5))
plt.plot(spectral_df["window_center"], spectral_df["spectral_entropy"])
plt.xlabel("Window center")
plt.ylabel("Entropy")
plt.title("Notebook 19 — Spectral Entropy")
savefig("notebook19_spectral_entropy.png")

# Mode rotation
plt.figure(figsize=(12,5))
plt.plot(drift_df["window_index"], drift_df["mode1_rotation_deg"], label="mode1")
plt.plot(drift_df["window_index"], drift_df["mode2_rotation_deg"], label="mode2")
plt.xlabel("Rolling block index")
plt.ylabel("Rotation (degrees)")
plt.title("Notebook 19 — Mode Rotation Angles")
plt.legend()
savefig("notebook19_mode_rotation_angles.png")

# Eigenspace similarity
plt.figure(figsize=(12,5))
plt.plot(drift_df["window_index"], drift_df["mode1_similarity"], label="mode1")
plt.plot(drift_df["window_index"], drift_df["mode2_similarity"], label="mode2")
plt.xlabel("Rolling block index")
plt.ylabel("Cosine similarity")
plt.title("Notebook 19 — Eigenspace Cosine Drift")
plt.legend()
savefig("notebook19_eigenspace_cosine_drift.png")

# Transition heatmap
plt.figure(figsize=(8,7))
plt.imshow(transition_matrix, aspect="auto")
plt.colorbar(label="Cosine similarity")
plt.xlabel("Window index")
plt.ylabel("Window index")
plt.title("Notebook 19 — Transition Heatmap")
savefig("notebook19_transition_heatmap.png")

# Phase embedding
plt.figure(figsize=(8,6))
plt.scatter(
    embedding_df["x"],
    embedding_df["y"],
    c=embedding_df["window_center"],
    s=12,
)
plt.xlabel("Embedding X")
plt.ylabel("Embedding Y")
plt.title("Notebook 19 — Phase Cluster Embedding")
savefig("notebook19_phase_cluster_embedding.png")

# Rolling lane heatmap
plt.figure(figsize=(11,6))
plt.imshow(X, aspect="auto")
plt.xticks(range(len(lane_cols)), [c.replace("lane_", "") for c in lane_cols])
plt.colorbar(label="Prime count")
plt.xlabel("Residue lane")
plt.ylabel("Rolling window")
plt.title("Notebook 19 — Rolling Prime Manifold")
savefig("notebook19_rolling_prime_manifold.png")


## Build Markdown report

In [ ]:
report_path = REPORTS_DIR / "report_19_temporal_spectral_dynamics.md"

output_links = "\n".join([
    '- Rolling vectors CSV: <a href="../results/notebook19_rolling_vectors.csv">`results/notebook19_rolling_vectors.csv`</a>',
    '- Spectral metrics CSV: <a href="../results/notebook19_rolling_spectral_metrics.csv">`results/notebook19_rolling_spectral_metrics.csv`</a>',
    '- Eigenspace drift CSV: <a href="../results/notebook19_eigenspace_drift.csv">`results/notebook19_eigenspace_drift.csv`</a>',
    '- Transition matrix CSV: <a href="../results/notebook19_transition_similarity_matrix.csv">`results/notebook19_transition_similarity_matrix.csv`</a>',
    '- Phase embedding CSV: <a href="../results/notebook19_phase_embedding.csv">`results/notebook19_phase_embedding.csv`</a>',
    '- Summary JSON: <a href="../results/notebook19_temporal_spectral_summary.json">`results/notebook19_temporal_spectral_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 19 — Temporal Spectral Dynamics of Residue Manifolds

Notebook 19 studies rolling eigenspace stability and temporal phase structure across mod30 residue manifolds.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Rolling windows | {len(rolling_df)} |
| Rolling spectral blocks | {len(spectral_df)} |
| Mean spectral entropy | {spectral_df["spectral_entropy"].mean():.6f} |
| Mean Mode 1 variance | {spectral_df["mode_1_variance"].mean():.6f} |
| Mean Mode 2 variance | {spectral_df["mode_2_variance"].mean():.6f} |
| Mean Mode 1 rotation | {drift_df["mode1_rotation_deg"].mean():.6f} |
| Mean Mode 2 rotation | {drift_df["mode2_rotation_deg"].mean():.6f} |

## Interpretation

- Rolling windows generate evolving residue manifolds.
- Local eigenspaces remain coherent but rotate gradually.
- Spectral entropy tracks manifold concentration vs fragmentation.
- Transition heatmaps reveal persistent temporal structure.
- Low-dimensional embeddings expose recurring manifold phases.

## Next step

Notebook 23 can introduce graph manifold embeddings and lane-transition operators.
"""

report_path.write_text(report)

print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:4000])


## Render figures

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook19_temporal_spectral_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:

    for p in RESULTS_DIR.glob("notebook19_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

    for p in FIGURES_DIR.glob("notebook19_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

    for p in REPORTS_DIR.glob("report_19_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# from google.colab import files
# files.download(str(export_path))
